# IPO Success Prediction – Notebook 02

## Download NSE Equity Bhavcopy Files

**Objective**
- Programmatically download NSE Equity Bhavcopy (Cash Market) files
- Use official NSE archives with proper request headers
- Store and unzip files locally for later price extraction

**Note**
- This notebook performs ONLY data downloading
- No price extraction or return computation is done here


In [16]:
import requests
import zipfile
import calendar
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
BHAV_ROOT = PROJECT_ROOT / "data" / "raw" / "bhavcopy"
BHAV_ROOT.mkdir(parents=True, exist_ok=True)

BHAV_ROOT


PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/bhavcopy')

In [17]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
})


In [18]:
def download_bhavcopy_day(year, month_str, date_str):
    year_dir = BHAV_ROOT / str(year)
    year_dir.mkdir(exist_ok=True)

    url = f"https://archives.nseindia.com/content/historical/EQUITIES/{year}/{month_str}/cm{date_str}bhav.csv.zip"
    zip_path = year_dir / f"cm{date_str}bhav.csv.zip"

    r = session.get(url, timeout=30)

    if r.status_code != 200:
        return False

    with open(zip_path, "wb") as f:
        f.write(r.content)

    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(year_dir)
    except zipfile.BadZipFile:
        zip_path.unlink(missing_ok=True)
        return False

    return True


In [19]:
def download_bhavcopy_month(year, month):
    month_str = calendar.month_abbr[month].upper()
    days = calendar.monthrange(year, month)[1]

    success = 0
    for day in range(1, days + 1):
        date_str = f"{day:02d}{month_str}{year}"
        if download_bhavcopy_day(year, month_str, date_str):
            success += 1

    print(f"Downloaded {success} bhavcopies for {month_str}-{year}")


In [20]:
download_bhavcopy_month(2012, 9)

list((BHAV_ROOT / "2012").glob("*.csv"))[:5]


Downloaded 20 bhavcopies for SEP-2012


[PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/bhavcopy/2012/cm25SEP2012bhav.csv'),
 PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/bhavcopy/2012/cm24SEP2012bhav.csv'),
 PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/bhavcopy/2012/cm21SEP2012bhav.csv'),
 PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/bhavcopy/2012/cm26SEP2012bhav.csv'),
 PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/raw/bhavcopy/2012/cm27SEP2012bhav.csv')]